Yeison Estibel Guerra Soto

In [0]:

# PUNTO 1. INDICADORES NACIONALES DEL MERCADO LABORAL


import pandas as pd

# 1. CARGA DE LAS TABLAS DELTA

ft = spark.table("workspace.electiva.fuerzadetrabajo").toPandas()
cgse = spark.table("workspace.electiva.cgse").toPandas()
ocupados = spark.table("workspace.electiva.ocupados").toPandas()
desempleados = spark.table("workspace.electiva.desempleados").toPandas()

# 2. VERIFICACIÓN DE LAS VARIABLES NECESARIAS

variables_requeridas = {
    "fuerza de trabajo": (ft, ["PET", "FEX_C18"]),
    "características generales": (cgse, ["FEX_C18"]),
    "ocupados": (ocupados, ["OCI", "FEX_C18"]),
    "desempleados": (desempleados, ["DSI", "FEX_C18"])
}

for nombre, (df, columnas) in variables_requeridas.items():
    faltantes = [col for col in columnas if col not in df.columns]
    if faltantes:
        raise ValueError(
            f"En la tabla de {nombre} faltan estas variables: {faltantes}. "
            f"Columnas disponibles: {df.columns.tolist()}"
        )

# 3. CONVERSIÓN DE VARIABLES A FORMATO NUMÉRICO

for df, columnas in [
    (ft, ["PET", "FEX_C18"]),
    (cgse, ["FEX_C18"]),
    (ocupados, ["OCI", "FEX_C18"]),
    (desempleados, ["DSI", "FEX_C18"])
]:
    for columna in columnas:
        df[columna] = pd.to_numeric(df[columna], errors="coerce")

# 4. ESTIMACIÓN DE LA POBLACIÓN EN EDAD DE TRABAJAR (PET)

pet = ft.loc[
    (ft["PET"] == 1) & (ft["FEX_C18"] > 0),
    "FEX_C18"
].sum()

# Población total representada por la encuesta
poblacion_total = cgse.loc[
    cgse["FEX_C18"] > 0, "FEX_C18"
].sum()

# Población que se encuentra fuera de la PET
poblacion_fuera_pet = poblacion_total - pet

# 5. ESTIMACIÓN DE OCUPADOS Y DESOCUPADOS

ocupados_total = ocupados.loc[
    (ocupados["OCI"] == 1) & (ocupados["FEX_C18"] > 0),
    "FEX_C18"
].sum()

desocupados_total = desempleados.loc[
    (desempleados["DSI"] == 1) & (desempleados["FEX_C18"] > 0),
    "FEX_C18"
].sum()

# 6. CÁLCULO DE LA PEA Y LA PEI

pea = ocupados_total + desocupados_total
pei = pet - pea

# 7. CÁLCULO DE LAS TASAS NACIONALES

tasa_ocupacion = (
    ocupados_total / pet * 100
    if pet > 0 else float("nan")
)

tasa_desempleo = (
    desocupados_total / pea * 100
    if pea > 0 else float("nan")
)

# 8. ORGANIZACIÓN Y PRESENTACIÓN DE LOS RESULTADOS

resultados_punto1 = pd.DataFrame({
    "Indicador": [
        "Población total estimada",
        "Población en edad de trabajar (PET)",
        "Población fuera de la PET",
        "Población económicamente activa (PEA)",
        "Población económicamente inactiva (PEI)",
        "Población ocupada",
        "Población desocupada",
        "Tasa de ocupación (TO)",
        "Tasa de desempleo (TD)"
    ],
    "Resultado": [
        poblacion_total,
        pet,
        poblacion_fuera_pet,
        pea,
        pei,
        ocupados_total,
        desocupados_total,
        tasa_ocupacion,
        tasa_desempleo
    ],
    "Unidad": [
        "Personas",
        "Personas",
        "Personas",
        "Personas",
        "Personas",
        "Personas",
        "Personas",
        "%",
        "%"
    ]
})

# Formato legible para presentar las cifras
resultados_punto1["Resultado"] = resultados_punto1.apply(
    lambda fila: (
        f"{fila['Resultado']:,.2f}%"
        if fila["Unidad"] == "%"
        else f"{fila['Resultado']:,.0f}"
    ),
    axis=1
)

display(resultados_punto1)

# 9. RESUMEN DE LOS INDICADORES PRINCIPALES

print("RESUMEN NACIONAL DEL MERCADO LABORAL")
print("-" * 45)
print(f"PET: {pet:,.0f} personas")
print(f"PEA: {pea:,.0f} personas")
print(f"PEI: {pei:,.0f} personas")
print(f"Ocupados: {ocupados_total:,.0f} personas")
print(f"Desocupados: {desocupados_total:,.0f} personas")
print(f"Tasa de ocupación: {tasa_ocupacion:.2f}%")
print(f"Tasa de desempleo: {tasa_desempleo:.2f}%")


Los resultados obtenidos permiten identificar la situación general del mercado laboral en Colombia durante junio de 2026. La Población en Edad de Trabajar (PET) fue de 41.202.631 personas, mientras que la población que no se encontraba en edad de trabajar fue de 11.346.181 personas. Dentro de la PET, la Población Económicamente Activa (PEA) alcanzó 26.593.429 personas, mientras que la Población Económicamente Inactiva (PEI) fue de 14.609.202 personas.
En cuanto a los indicadores laborales, la Tasa de Ocupación se ubicó en 59,36 %, lo que indica la proporción de personas en edad de trabajar que se encontraban ocupadas. Por su parte, la Tasa de Desempleo fue de 8,03 %, lo que significa que este porcentaje de la población económicamente activa se encontraba buscando trabajo sin conseguirlo.
En general, los resultados muestran que la mayoría de la población económicamente activa se encontraba ocupada, aunque todavía existía una proporción de personas que no lograba incorporarse al mercado laboral.

In [0]:


# PUNTO 2. TASA DE DESEMPLEO POR DEPARTAMENTO Y SEXO

import pandas as pd
import numpy as np

# 1. CARGA DE LAS TABLAS DELTA

ft = spark.table("workspace.electiva.fuerzadetrabajo").toPandas()
cgse = spark.table("workspace.electiva.cgse").toPandas()
ocupados = spark.table("workspace.electiva.ocupados").toPandas()
desempleados = spark.table("workspace.electiva.desempleados").toPandas()

# 2. IDENTIFICACIÓN DE LAS PERSONAS

llaves = ["DIRECTORIO", "SECUENCIA_P", "ORDEN"]

# Verificar las variables requeridas
for nombre, df, columnas in [
    ("CGSE", cgse, llaves + ["P3271"]),
    ("Ocupados", ocupados, llaves + ["DPTO", "OCI", "FEX_C18"]),
    ("Desempleados", desempleados, llaves + ["DPTO", "DSI", "FEX_C18"])
]:
    faltantes = [c for c in columnas if c not in df.columns]
    if faltantes:
        raise ValueError(
            f"Faltan columnas en {nombre}: {faltantes}"
        )

# Conservar la identificación y el sexo de cada persona
sexo_persona = cgse[llaves + ["P3271"]].drop_duplicates(
    subset=llaves
)

# 3. PREPARACIÓN DE OCUPADOS Y DESOCUPADOS

# Incorporar el sexo a los registros de ocupados
base_ocupados = ocupados.loc[
    ocupados["OCI"] == 1
].merge(
    sexo_persona,
    on=llaves,
    how="inner",
    validate="many_to_one"
)

# Incorporar el sexo a los registros de desocupados
base_desocupados = desempleados.loc[
    desempleados["DSI"] == 1
].merge(
    sexo_persona,
    on=llaves,
    how="inner",
    validate="many_to_one"
)

# Convertir las variables necesarias a formato numérico
for df in [base_ocupados, base_desocupados]:
    df["FEX_C18"] = pd.to_numeric(
        df["FEX_C18"], errors="coerce"
    )
    df["DPTO"] = pd.to_numeric(
        df["DPTO"], errors="coerce"
    )
    df["P3271"] = pd.to_numeric(
        df["P3271"], errors="coerce"
    )

# Mantener registros con información válida
base_ocupados = base_ocupados.loc[
    (base_ocupados["FEX_C18"] > 0) &
    base_ocupados["DPTO"].notna() &
    base_ocupados["P3271"].isin([1, 2])
].copy()

base_desocupados = base_desocupados.loc[
    (base_desocupados["FEX_C18"] > 0) &
    base_desocupados["DPTO"].notna() &
    base_desocupados["P3271"].isin([1, 2])
].copy()

# 4. CÁLCULO DE LOS OCUPADOS Y DESOCUPADOS
#    POR DEPARTAMENTO Y SEXO

suma_ocupados = (
    base_ocupados
    .groupby(["DPTO", "P3271"])["FEX_C18"]
    .sum()
    .rename("Ocupados")
)

suma_desocupados = (
    base_desocupados
    .groupby(["DPTO", "P3271"])["FEX_C18"]
    .sum()
    .rename("Desocupados")
)

# Unir las estimaciones de ambos grupos
tabla_sexo = pd.concat(
    [suma_ocupados, suma_desocupados],
    axis=1
).fillna(0).reset_index()

# 5. CÁLCULO DE LA PEA Y LA TASA DE DESEMPLEO

tabla_sexo["PEA"] = (
    tabla_sexo["Ocupados"] +
    tabla_sexo["Desocupados"]
)

tabla_sexo["Tasa_desempleo"] = np.where(
    tabla_sexo["PEA"] > 0,
    tabla_sexo["Desocupados"] /
    tabla_sexo["PEA"] * 100,
    np.nan
)

tabla_sexo["Sexo"] = tabla_sexo["P3271"].map({
    1: "Hombres",
    2: "Mujeres"
})

# 6. IDENTIFICACIÓN DE LOS DEPARTAMENTOS

nombres_departamentos = {
    5: "Antioquia",
    8: "Atlántico",
    11: "Bogotá D.C.",
    13: "Bolívar",
    15: "Boyacá",
    17: "Caldas",
    18: "Caquetá",
    19: "Cauca",
    20: "Cesar",
    23: "Córdoba",
    25: "Cundinamarca",
    27: "Chocó",
    41: "Huila",
    44: "La Guajira",
    47: "Magdalena",
    50: "Meta",
    52: "Nariño",
    54: "Norte de Santander",
    63: "Quindío",
    66: "Risaralda",
    68: "Santander",
    70: "Sucre",
    73: "Tolima",
    76: "Valle del Cauca",
    81: "Arauca",
    85: "Casanare",
    86: "Putumayo",
    88: "San Andrés y Providencia",
    91: "Amazonas",
    94: "Guainía",
    95: "Guaviare",
    97: "Vaupés",
    99: "Vichada"
}

tabla_sexo["Departamento"] = (
    tabla_sexo["DPTO"].astype(int)
    .map(nombres_departamentos)
    .fillna("Código departamental no identificado")
)

# 7. TABLA COMPARATIVA DE HOMBRES Y MUJERES

comparacion_sexo = tabla_sexo.pivot_table(
    index="Departamento",
    columns="Sexo",
    values="Tasa_desempleo",
    aggfunc="first"
).reset_index()

# 8. TASA TOTAL DE DESEMPLEO POR DEPARTAMENTO

totales_departamento = (
    tabla_sexo
    .groupby("Departamento")[
        ["Ocupados", "Desocupados", "PEA"]
    ]
    .sum()
    .reset_index()
)

totales_departamento["Tasa_total"] = np.where(
    totales_departamento["PEA"] > 0,
    totales_departamento["Desocupados"] /
    totales_departamento["PEA"] * 100,
    np.nan
)

# Incorporar la tasa total a la comparación por sexo
resultado_punto2 = comparacion_sexo.merge(
    totales_departamento[
        ["Departamento", "Tasa_total"]
    ],
    on="Departamento",
    how="left"
)

resultado_punto2 = resultado_punto2.rename(columns={
    "Hombres": "Desempleo hombres (%)",
    "Mujeres": "Desempleo mujeres (%)",
    "Tasa_total": "Desempleo total (%)"
})

# Ordenar por nombre del departamento
resultado_punto2 = resultado_punto2.sort_values(
    "Departamento"
).reset_index(drop=True)

print("TASA DE DESEMPLEO POR DEPARTAMENTO Y SEXO")

display(
    resultado_punto2.round(2)
)

# 9. DEPARTAMENTOS CON MAYOR Y MENOR DESEMPLEO TOTAL

departamentos_validos = totales_departamento.loc[
    (totales_departamento["PEA"] > 0) &
    totales_departamento["Tasa_total"].notna()
].copy()

mayor_desempleo = departamentos_validos.nlargest(
    3, "Tasa_total"
)[["Departamento", "Tasa_total"]]

menor_desempleo = departamentos_validos.nsmallest(
    3, "Tasa_total"
)[["Departamento", "Tasa_total"]]

print("Tres departamentos con mayor tasa de desempleo:")
display(mayor_desempleo.round(2))

print("Tres departamentos con menor tasa de desempleo:")
display(menor_desempleo.round(2))

# 10. RESUMEN NACIONAL POR SEXO

resumen_sexo = (
    tabla_sexo
    .groupby("Sexo")[["Ocupados", "Desocupados", "PEA"]]
    .sum()
    .reset_index()
)

resumen_sexo["Tasa_desempleo"] = np.where(
    resumen_sexo["PEA"] > 0,
    resumen_sexo["Desocupados"] /
    resumen_sexo["PEA"] * 100,
    np.nan
)

print("RESUMEN NACIONAL POR SEXO")
display(resumen_sexo.round(2))


Los resultados permiten observar diferencias en la tasa de desempleo entre los departamentos de Colombia y entre hombres y mujeres. Al considerar el total de la población, Putumayo presentó la tasa de desempleo más alta, con 22,18 %, mientras que Vaupés registró la más baja, con 2,46 %.
Al analizar los resultados por sexo, se encontró que para los hombres, la tasa de desempleo más alta correspondió a San Andrés con 20,98 %, mientras que la más baja se presentó en Vaupés, con 3,00 %. En el caso de las mujeres, Putumayo registró la tasa más alta con 25,02 %, y Vaupés presentó la más baja con 1,78 %.
Estos resultados muestran que el desempleo no se distribuye de manera uniforme en el territorio nacional. Además, las diferencias entre hombres y mujeres indican que la situación laboral puede variar según el sexo y el departamento.

In [0]:


# PUNTO 3. TASA DE DESEMPLEO POR DEPARTAMENTO Y GRUPO DE EDAD

import pandas as pd
import numpy as np

# 1. CARGAR LAS TABLAS DELTA

ft = spark.table(
    "workspace.electiva.fuerzadetrabajo"
).toPandas()

cgse = spark.table(
    "workspace.electiva.cgse"
).toPandas()

ocupados = spark.table(
    "workspace.electiva.ocupados"
).toPandas()

desempleados = spark.table(
    "workspace.electiva.desempleados"
).toPandas()

# 2. DEFINIR LAS LLAVES Y COMPROBAR LAS VARIABLES

llaves = ["DIRECTORIO", "SECUENCIA_P", "ORDEN"]

for nombre, df, columnas in [
    ("CGSE", cgse, llaves + ["P6040"]),
    ("Ocupados", ocupados, llaves + ["DPTO", "OCI", "FEX_C18"]),
    ("Desempleados", desempleados, llaves + ["DPTO", "DSI", "FEX_C18"])
]:
    faltantes = [c for c in columnas if c not in df.columns]

    if faltantes:
        raise ValueError(
            f"En la tabla {nombre} faltan las columnas: {faltantes}"
        )

# 3. PREPARAR LA EDAD DE CADA PERSONA

edad_persona = cgse[llaves + ["P6040"]].copy()

edad_persona["P6040"] = pd.to_numeric(
    edad_persona["P6040"], errors="coerce"
)

# Evitar duplicar personas al hacer las uniones
edad_persona = edad_persona.drop_duplicates(
    subset=llaves
)

# 4. CLASIFICAR POR GRUPOS DE EDAD

def clasificar_edad(edad):
    if pd.isna(edad):
        return None
    elif 18 <= edad < 24:
        return "18 a 24"
    elif 24 <= edad < 30:
        return "24 a 30"
    elif 30 <= edad < 40:
        return "30 a 40"
    elif 40 <= edad < 50:
        return "40 a 50"
    elif 50 <= edad < 60:
        return "50 a 60"
    elif edad >= 60:
        return "60 en adelante"
    else:
        return None

edad_persona["Grupo_Edad"] = edad_persona["P6040"].apply(
    clasificar_edad
)

# 5. UNIR LOS OCUPADOS CON LA EDAD

oc_p3 = ocupados.loc[
    ocupados["OCI"] == 1
].merge(
    edad_persona[llaves + ["Grupo_Edad"]],
    on=llaves,
    how="inner",
    validate="many_to_one"
)

# 6. UNIR LOS DESOCUPADOS CON LA EDAD

des_p3 = desempleados.loc[
    desempleados["DSI"] == 1
].merge(
    edad_persona[llaves + ["Grupo_Edad"]],
    on=llaves,
    how="inner",
    validate="many_to_one"
)

# 7. LIMPIAR LOS REGISTROS UTILIZADOS

for df in [oc_p3, des_p3]:
    df["DPTO"] = pd.to_numeric(
        df["DPTO"], errors="coerce"
    )
    df["FEX_C18"] = pd.to_numeric(
        df["FEX_C18"], errors="coerce"
    )

oc_p3 = oc_p3.loc[
    oc_p3["Grupo_Edad"].notna() &
    oc_p3["DPTO"].notna() &
    (oc_p3["FEX_C18"] > 0)
].copy()

des_p3 = des_p3.loc[
    des_p3["Grupo_Edad"].notna() &
    des_p3["DPTO"].notna() &
    (des_p3["FEX_C18"] > 0)
].copy()

# 8. ESTIMAR OCUPADOS Y DESOCUPADOS POR GRUPO

ocupados_edad = (
    oc_p3.groupby(["DPTO", "Grupo_Edad"])["FEX_C18"]
    .sum()
    .rename("Ocupados")
)

desocupados_edad = (
    des_p3.groupby(["DPTO", "Grupo_Edad"])["FEX_C18"]
    .sum()
    .rename("Desocupados")
)

tabla_edad = pd.concat(
    [ocupados_edad, desocupados_edad],
    axis=1
).fillna(0).reset_index()

# 9. CALCULAR PEA Y TASA DE DESEMPLEO

tabla_edad["PEA"] = (
    tabla_edad["Ocupados"] +
    tabla_edad["Desocupados"]
)

tabla_edad["Tasa_Desempleo"] = np.where(
    tabla_edad["PEA"] > 0,
    tabla_edad["Desocupados"] /
    tabla_edad["PEA"] * 100,
    np.nan
)

# 10. ASIGNAR LOS NOMBRES DE LOS DEPARTAMENTOS

dpto_nombre = {
    5: "ANTIOQUIA",
    8: "ATLANTICO",
    11: "BOGOTA",
    13: "BOLIVAR",
    15: "BOYACA",
    17: "CALDAS",
    18: "CAQUETA",
    19: "CAUCA",
    20: "CESAR",
    23: "CORDOBA",
    25: "CUNDINAMARCA",
    27: "CHOCO",
    41: "HUILA",
    44: "GUAJIRA",
    47: "MAGDALENA",
    50: "META",
    52: "NARIÑO",
    54: "NORTE DE SANTANDER",
    63: "QUINDIO",
    66: "RISARALDA",
    68: "SANTANDER",
    70: "SUCRE",
    73: "TOLIMA",
    76: "VALLE DEL CAUCA",
    81: "ARAUCA",
    85: "CASANARE",
    86: "PUTUMAYO",
    88: "SAN ANDRES",
    91: "AMAZONAS",
    94: "GUAINIA",
    95: "GUAVIARE",
    97: "VAUPES",
    99: "VICHADA"
}

tabla_edad["Departamento"] = (
    tabla_edad["DPTO"].astype(int).map(dpto_nombre)
)

# Excluir los registros cuyo departamento no pudo identificarse
tabla_edad = tabla_edad.loc[
    tabla_edad["Departamento"].notna()
].copy()

# 11. IDENTIFICAR EL GRUPO CON MAYOR Y MENOR DESEMPLEO
#     EN CADA DEPARTAMENTO

tabla_valida = tabla_edad.loc[
    (tabla_edad["PEA"] > 0) &
    tabla_edad["Tasa_Desempleo"].notna()
].copy()

idx_mayor = tabla_valida.groupby(
    "Departamento"
)["Tasa_Desempleo"].idxmax()

idx_menor = tabla_valida.groupby(
    "Departamento"
)["Tasa_Desempleo"].idxmin()

mayor_td = tabla_valida.loc[
    idx_mayor,
    ["Departamento", "Grupo_Edad", "Tasa_Desempleo"]
].rename(columns={
    "Grupo_Edad": "Grupo con mayor desempleo",
    "Tasa_Desempleo": "Mayor TD (%)"
})

menor_td = tabla_valida.loc[
    idx_menor,
    ["Departamento", "Grupo_Edad", "Tasa_Desempleo"]
].rename(columns={
    "Grupo_Edad": "Grupo con menor desempleo",
    "Tasa_Desempleo": "Menor TD (%)"
})

resultado_punto3 = mayor_td.merge(
    menor_td,
    on="Departamento",
    how="inner"
)

resultado_punto3 = resultado_punto3.sort_values(
    "Departamento"
).reset_index(drop=True)

# 12. PRESENTAR LOS RESULTADOS

print(
    "PUNTO 3: GRUPOS DE EDAD CON MAYOR Y MENOR "
    "DESEMPLEO POR DEPARTAMENTO"
)

display(resultado_punto3.round(2))

# 13. TABLA COMPLETA PARA COMPARAR TODOS LOS GRUPOS

tabla_completa = tabla_valida[
    [
        "Departamento",
        "Grupo_Edad",
        "Ocupados",
        "Desocupados",
        "PEA",
        "Tasa_Desempleo"
    ]
].sort_values(
    ["Departamento", "Grupo_Edad"]
).reset_index(drop=True)

print("TASA DE DESEMPLEO PARA CADA GRUPO DE EDAD")
display(tabla_completa.round(2))

# 14. RESUMEN GENERAL DE LOS RESULTADOS

print(
    "Grupo de edad con mayor desempleo en más departamentos:"
)

print(
    tabla_valida.loc[
        tabla_valida.groupby("Departamento")[
            "Tasa_Desempleo"
        ].idxmax(),
        "Grupo_Edad"
    ].value_counts()
)

print(
    "\nGrupo de edad con menor desempleo en más departamentos:"
)

print(
    tabla_valida.loc[
        tabla_valida.groupby("Departamento")[
            "Tasa_Desempleo"
        ].idxmin(),
        "Grupo_Edad"
    ].value_counts()
)


Los resultados muestran diferencias en la tasa de desempleo según la edad y el departamento. En general, los valores más altos se concentraron en el grupo de 18 a 24 años, aunque en algunos departamentos destacó el grupo de 24 a 30 años. Grupo de edad con mayor desempleo en más departamentos:
Grupo Edad
18 a 24 = 25,
24 a 30 = 7,
40 a 50 = 1.
San Andrés presentó la tasa más alta, con 33,66 % para las personas de 18 a 24 años, seguido de Sucre, con 33,14 %, y Vichada, con 31,6 %. Por su parte, Putumayo registró un 30,58 % en el grupo de 24 a 30 años, mientras que Casanare alcanzó el 29,16 % entre los jóvenes de 18 a 24 años.
En contraste, las tasas más bajas incluyeron valores de 0 % en Amazonas, Casanare y Guainía, según sus respectivos grupos de edad. También se destacaron Risaralda, con 0,33 % para las personas de 60 años en adelante, y Caquetá, con 0,56 % en el grupo de 50 a 60 años.
En conjunto, los resultados evidencian que el desempleo afecta especialmente a los grupos jóvenes en varios departamentos, mientras que las tasas suelen ser menores en los grupos de mayor edad.

In [0]:

# PUNTO 4. MENOR DESEMPLEO EN MUJERES DE 18 A 25 AÑOS

import pandas as pd
import numpy as np

# 1. Cargar las tablas Delta
cgse = spark.table(
    "workspace.electiva.cgse"
).toPandas()

ocupados = spark.table(
    "workspace.electiva.ocupados"
).toPandas()

desempleados = spark.table(
    "workspace.electiva.desempleados"
).toPandas()

# 2. Identificadores de las personas
llaves = ["DIRECTORIO", "SECUENCIA_P", "ORDEN"]

# Verificar las variables necesarias
for nombre, df, columnas in [
    ("CGSE", cgse, llaves + ["P3271", "P6040"]),
    ("Ocupados", ocupados, llaves + ["DPTO", "OCI", "FEX_C18"]),
    ("Desempleados", desempleados, llaves + ["DPTO", "DSI", "FEX_C18"])
]:
    faltantes = [c for c in columnas if c not in df.columns]
    if faltantes:
        raise ValueError(
            f"Faltan columnas en {nombre}: {faltantes}"
        )

# 3. Preparar sexo y edad
personas = cgse[llaves + ["P3271", "P6040"]].copy()
personas = personas.drop_duplicates(subset=llaves)

personas["P3271"] = pd.to_numeric(
    personas["P3271"], errors="coerce"
)
personas["P6040"] = pd.to_numeric(
    personas["P6040"], errors="coerce"
)

# 4. Filtrar mujeres de 18 a 25 años
mujeres_18_25 = personas.loc[
    (personas["P3271"] == 2) &
    (personas["P6040"].between(18, 25))
].copy()

# 5. Unir los datos de ocupados y desocupados
oc = ocupados.loc[
    ocupados["OCI"] == 1
].merge(
    mujeres_18_25[llaves],
    on=llaves,
    how="inner",
    validate="many_to_one"
)

des = desempleados.loc[
    desempleados["DSI"] == 1
].merge(
    mujeres_18_25[llaves],
    on=llaves,
    how="inner",
    validate="many_to_one"
)

# 6. Limpiar los factores de expansión
for df in [oc, des]:
    df["DPTO"] = pd.to_numeric(
        df["DPTO"], errors="coerce"
    )
    df["FEX_C18"] = pd.to_numeric(
        df["FEX_C18"], errors="coerce"
    )

oc = oc.loc[
    oc["DPTO"].notna() & (oc["FEX_C18"] > 0)
].copy()

des = des.loc[
    des["DPTO"].notna() & (des["FEX_C18"] > 0)
].copy()

# 7. Sumar ocupados y desocupados por departamento
oc_sum = oc.groupby("DPTO")["FEX_C18"].sum().rename("Ocupados")
des_sum = des.groupby("DPTO")["FEX_C18"].sum().rename("Desocupados")

tabla_p4 = pd.concat(
    [oc_sum, des_sum], axis=1
).fillna(0).reset_index()

# 8. Calcular la PEA y la tasa de desempleo
tabla_p4["PEA"] = (
    tabla_p4["Ocupados"] + tabla_p4["Desocupados"]
)

tabla_p4["Tasa_Desempleo"] = np.where(
    tabla_p4["PEA"] > 0,
    tabla_p4["Desocupados"] / tabla_p4["PEA"] * 100,
    np.nan
)

# 9. Asignar nombres a los departamentos
nombres_dpto = {
    5: "Antioquia",
    8: "Atlántico",
    11: "Bogotá D.C.",
    13: "Bolívar",
    15: "Boyacá",
    17: "Caldas",
    18: "Caquetá",
    19: "Cauca",
    20: "Cesar",
    23: "Córdoba",
    25: "Cundinamarca",
    27: "Chocó",
    41: "Huila",
    44: "La Guajira",
    47: "Magdalena",
    50: "Meta",
    52: "Nariño",
    54: "Norte de Santander",
    63: "Quindío",
    66: "Risaralda",
    68: "Santander",
    70: "Sucre",
    73: "Tolima",
    76: "Valle del Cauca",
    81: "Arauca",
    85: "Casanare",
    86: "Putumayo",
    88: "San Andrés",
    91: "Amazonas",
    94: "Guainía",
    95: "Guaviare",
    97: "Vaupés",
    99: "Vichada"
}

tabla_p4["Departamento"] = (
    tabla_p4["DPTO"].astype(int).map(nombres_dpto)
)

# 10. Obtener los tres departamentos con menor tasa
validos = tabla_p4.loc[
    (tabla_p4["PEA"] > 0) &
    tabla_p4["Tasa_Desempleo"].notna() &
    tabla_p4["Departamento"].notna()
].copy()

top3_menor_td_mujeres = (
    validos[
        ["Departamento", "Ocupados", "Desocupados",
         "PEA", "Tasa_Desempleo"]
    ]
    .sort_values("Tasa_Desempleo")
    .head(3)
    .reset_index(drop=True)
)

top3_menor_td_mujeres["Tasa_Desempleo"] = (
    top3_menor_td_mujeres["Tasa_Desempleo"].round(2)
)

print("TRES DEPARTAMENTOS CON MENOR DESEMPLEO FEMENINO")
print("Mujeres de 18 a 25 años")
display(top3_menor_td_mujeres)



Los tres departamentos con menor tasa de desempleo entre las mujeres de 18 a 25 años son:
1.Vaupés: 0,00 %.
2.Nariño: 4,29 %.
3.Amazonas: 4,87 %.
Vaupés presenta la tasa más baja, seguido de Nariño y Amazonas. En los dos últimos departamentos, las tasas se encuentran por debajo del promedio nacional de desempleo del 8,03 %.


In [0]:


# PUNTO 5. PROMEDIO DE INGRESO LABORAL POR DEPARTAMENTO


import pandas as pd
import numpy as np

# 1. Cargar la tabla Delta de ocupados
ocupados = spark.table(
    "workspace.electiva.ocupados"
).toPandas()

# 2. Verificar las variables necesarias
columnas_necesarias = [
    "DPTO", "OCI", "INGLABO", "FEX_C18"
]

faltantes = [
    c for c in columnas_necesarias
    if c not in ocupados.columns
]

if faltantes:
    raise ValueError(
        f"Faltan columnas en la tabla ocupados: {faltantes}"
    )

# 3. Seleccionar los cinco departamentos
departamentos = {
    66: "Risaralda",
    17: "Caldas",
    63: "Quindío",
    5: "Antioquia",
    76: "Valle del Cauca"
}

# 4. Conservar solamente personas ocupadas
ingresos = ocupados.loc[
    (ocupados["OCI"] == 1) &
    (pd.to_numeric(ocupados["DPTO"], errors="coerce")
     .isin(departamentos.keys()))
].copy()

# 5. Convertir las variables a formato numérico
ingresos["INGLABO"] = pd.to_numeric(
    ingresos["INGLABO"], errors="coerce"
)

ingresos["FEX_C18"] = pd.to_numeric(
    ingresos["FEX_C18"], errors="coerce"
)

# 6. Eliminar ingresos no válidos y factores no positivos
ingresos = ingresos.loc[
    ingresos["INGLABO"].notna() &
    ingresos["FEX_C18"].notna() &
    (ingresos["INGLABO"] > 0) &
    (ingresos["FEX_C18"] > 0)
].copy()

# 7. Calcular el ingreso ponderado
ingresos["Ingreso_Ponderado"] = (
    ingresos["INGLABO"] * ingresos["FEX_C18"]
)

# 8. Sumar los ingresos ponderados y los factores
tabla_p5 = (
    ingresos.groupby("DPTO")[
        ["Ingreso_Ponderado", "FEX_C18"]
    ]
    .sum()
    .reset_index()
)

# 9. Calcular el promedio ponderado
tabla_p5["Promedio_INGLABO"] = (
    tabla_p5["Ingreso_Ponderado"] /
    tabla_p5["FEX_C18"]
)

# 10. Incorporar los nombres de los departamentos
tabla_p5["Departamento"] = (
    tabla_p5["DPTO"].map(departamentos)
)

# 11. Organizar los resultados de mayor a menor
resultado_punto5 = (
    tabla_p5[
        ["Departamento", "Promedio_INGLABO"]
    ]
    .sort_values(
        "Promedio_INGLABO",
        ascending=False
    )
    .reset_index(drop=True)
)

# 12. Mostrar los resultados en pesos colombianos
resultado_punto5["Promedio_INGLABO"] = (
    resultado_punto5["Promedio_INGLABO"].round(0)
)

print("PROMEDIO DE INGRESO LABORAL MENSUAL POR DEPARTAMENTO")
display(resultado_punto5)

# 13. Presentar también los valores en formato monetario
print("\nINGRESO PROMEDIO MENSUAL (COP)")
for _, fila in resultado_punto5.iterrows():
    print(
        f"{fila['Departamento']}: "
        f"${fila['Promedio_INGLABO']:,.0f}"
    )


El promedio ponderado del ingreso laboral mensual en los cinco departamentos analizados presenta los siguientes resultados:
Antioquia: $2.486.207.
Risaralda: $2.234.061.
Caldas: $2.226.450.
Quindío: $2.171.560.
Valle del Cauca: $2.147.996.
Antioquia registra el mayor ingreso promedio, mientras que Valle del Cauca presenta el menor de los cinco departamentos. La diferencia entre ambos es de aproximadamente $338.211 mensuales.
En general, los resultados muestran que los ingresos laborales promedio se sitúan alrededor de los $2,2 millones mensuales, aunque existen diferencias entre los departamentos.